# Copy Concepts — Session 03: Duplicate, Yet Not a Duplicate

Copying looks trivial until data is nested. Assigning a list to a new name does not copy it — it creates a second reference to the same object. Even `copy()` and slicing only copy the outer container, leaving the inner objects shared. This notebook makes those distinctions precise with live `id()` experiments.

**The big picture.** In Python, names are labels attached to objects, and assignment attaches another label to the same object. A **shallow copy** creates a new outer container whose contents still point at the original inner objects. A **deep copy** recursively clones the whole tree so that nothing is shared.

**Why it matters.** The nested-copy mistake is one of the most common sources of mysterious bugs: you edit a copy, and the original changes too. Understanding aliasing, shallow copy, and deep copy lets you reason about function arguments, configuration objects, and any nested structure.

**Roadmap.** In this notebook you will:
1. Prove that `b = a` is aliasing, not copying, using `id()`.
2. Compare the three shallow-copy tools: `[:]`, `.copy()`, and `copy.copy()`.
3. Trigger the nested trap where a shallow copy shares its inner lists.
4. Fix it with `copy.deepcopy()`.
5. Learn the interview-grade test that distinguishes shallow from deep.

**Key definitions.** *Aliasing*: two names bound to one object, sharing a single `id()`. *Shallow copy*: a new outer object whose elements are the same references. *Deep copy*: a full recursive clone with fresh objects throughout the structure.

## 1. Aliasing vs Copying

Assignment never copies. `b = a` simply binds a second name to the same object, so `id(a) == id(b)` and any mutation through either name is visible through the other. This is **aliasing**, and it is exactly what you want for large objects — copying everything on every assignment would be wasteful.

| Term | How it is created | Object relationship |
|---|---|---|
| Aliasing | `b = a` | Same `id` — one object, two names |
| Shallow copy | `a.copy()`, `a[:]`, `copy.copy(a)` | Outer `id` differs, inner `id`s are shared |
| Deep copy | `copy.deepcopy(a)` | Outer and inner `id`s all differ |

The practical consequence: mutation of a mutable object is seen by every alias, while reassignment (`b = something_else`) only rebinds one name. The experiments that follow prove each row of this table with `id()`.

In [1]:
# 🐣 ALIASING — do nam, ek banda! (school fonts assignment danger zone 🚨)
a = [1, 2, 3]
b = a                     # COPY NAHI — sirf naya naam-tag!
print("a:", a, "| id:", id(a))
print("b:", b, "| id:", id(b), "← SAME ID! do naam, ek object")
b.append(99)
print("b.append(99) ke baad a:", a, "← DONO badle?! Haan — same object tha! 😱")
# Expected OUTPUT:
# a: [1, 2, 3] | id: <koi-number>
# b: [1, 2, 3] | id: <SAME-number> ← SAME ID! do naam, ek object
# b.append(99) ke baad a: [1, 2, 3, 99] ← DONO badle?! Haan — same object tha! 😱

a: [1, 2, 3] | id: 2896985427968
b: [1, 2, 3] | id: 2896985427968 ← SAME ID! do naam, ek object
b.append(99) ke baad a: [1, 2, 3, 99] ← DONO badle?! Haan — same object tha! 😱


## 2. Shallow Copy

A shallow copy builds a new outer container but reuses the inner references. For a **flat** list of numbers or strings that is completely safe: appending to the copy does not touch the original, because the two lists own separate pointer arrays.

All three common tools produce the same kind of result: slicing `original[:]`, the method `original.copy()`, and `copy.copy(original)`. Their outer `id()` values differ from the original, proving a new object was created.

The limitation appears only with nesting. If the elements are themselves mutable — lists, dictionaries, sets — the shallow copy still holds the *original* inner objects, so changes inside them leak to both the copy and the original. The next section demonstrates that trap.

In [2]:
# 🐥 SHALLOW copy — 3 tools, ek hi story! 🛋️↔️🛋️
original = [1, 2, 3]
c1 = original[:]                       # tool#1: slicing
c2 = original.copy()                   # tool#2: .copy()
import copy
c3 = copy.copy(original)               # tool#3: copy.copy()
print("original id:", id(original))
print("c1 id      :", id(c1), "← different (naya outer box ✅)")
print("c2 id      :", id(c2), "← different")
print("c3 id      :", id(c3), "← different")

# Flat list pe update safety — shallow YAHAN kaafi!
c1.append(99)
print("\nc1.append(99) → original:", original, "← SAFE! (flat list me shallow hi kaafi ✅)")
# Expected OUTPUT:
# original id: <n1>
# c1 id      : <n2> ← different (naya outer box ✅)
# c2 id      : <n3> ← different
# c3 id      : <n4> ← different
#
# c1.append(99) → original: [1, 2, 3] ← SAFE! (flat list me shallow hi kaafi ✅)

original id: 2896986205312
c1 id      : 2896986200000 ← different (naya outer box ✅)
c2 id      : 2896986203776 ← different
c3 id      : 2896986205056 ← different

c1.append(99) → original: [1, 2, 3] ← SAFE! (flat list me shallow hi kaafi ✅)


## 3. The Nested Trap

For a list of lists, a shallow copy is a half-measure. The outer list is new, but each inner list is shared. `id(matrix) != id(shallow)`, yet `id(matrix[0]) == id(shallow[0])`.

The consequence is surprising the first time: `shallow[0].append(99)` also changes `matrix[0]`, because both names point at the same inner list. The outer containers are separate, but their contents overlap.

| Operation | Outer new? | Inner new? |
|---|---|---|
| `matrix.copy()` | Yes | No |
| `copy.deepcopy(matrix)` | Yes | Yes |

The rule to remember: a shallow copy is safe when the contents are immutable or never mutated in place. As soon as nested mutable data can change, you need a deep copy.

### Semantics
Assignment creates references (aliasing). Shallow copy copies top-level; nested objects shared. Deep copy recursively copies all objects.

### Why matters
Prevents unintended side effects in mutable nested structures.

### When to use
Shallow for flat; deep when independence needed.

### Common mistakes
Assuming list slicing always gives deep copy for nested.

In [3]:
# 🐔 THE NESTED TRAP — shallow ka aaina-darr! 🤯⭐ (THE twist!)
matrix = [[1, 2, 3], [4, 5, 6]]
shallow = matrix.copy()                # outer NEW... par ANDAR?
print("outer ids differ:", id(matrix) != id(shallow), "✅")
print("INNER ids same? :", id(matrix[0]) == id(shallow[0]), "😱 INNER SHARED!!")

shallow[0].append(99)                  # inner furniture todte hain...
print("\nshallow  :", shallow)
print("matrix   :", matrix, "← MATRIX BHI BADAL GAYA?! stapled pages 💔")
# Expected OUTPUT:
# outer ids differ: True ✅
# INNER ids same? : True 😱 INNER SHARED!!
#
# shallow  : [[1, 2, 3, 99], [4, 5, 6]]
# matrix   : [[1, 2, 3, 99], [4, 5, 6]] ← MATRIX BHI BADAL GAYA?! stapled pages 💔

outer ids differ: True ✅
INNER ids same? : True 😱 INNER SHARED!!

shallow  : [[1, 2, 3, 99], [4, 5, 6]]
matrix   : [[1, 2, 3, 99], [4, 5, 6]] ← MATRIX BHI BADAL GAYA?! stapled pages 💔


## 4. Deep Copy

`copy.deepcopy(obj)` clones the object recursively: the outer container is new, and every nested mutable object is new as well. Mutating the deep copy therefore leaves the original completely untouched.

Because it walks the entire structure, `deepcopy` is slower and more memory-hungry than a shallow copy — a fair trade when correctness requires full independence. It also handles cycles and shared references correctly, preserving the shape of the original graph.

Use deepcopy when a structure contains nested mutables that may be modified, when you keep a pristine snapshot for comparison, or when passing configuration objects that called code might alter. For flat data, a shallow copy is faster and just as safe.

### Semantics
Assignment aliases; shallow copy copies top level (nested shared); deep copy copies recursively.

In [4]:
# 🐔 DEEP copy — full-page Xerox, poora azaad! ✂️
matrix2 = [[1, 2, 3], [4, 5, 6]]
deep = copy.deepcopy(matrix2)
deep[0].append(99)
print("deep     :", deep)
print("matrix2  :", matrix2, "← ORIGINAL bilkul untouched! 🎉")
print("inner ids different:", id(matrix2[0]) != id(deep[0]), "✅ poorna-freedom")
# Expected OUTPUT:
# deep     : [[1, 2, 3, 99], [4, 5, 6]]
# matrix2  : [[1, 2, 3], [4, 5, 6]] ← ORIGINAL bilkul untouched! 🎉
# inner ids different: True ✅ poorna-freedom

deep     : [[1, 2, 3, 99], [4, 5, 6]]
matrix2  : [[1, 2, 3], [4, 5, 6]] ← ORIGINAL bilkul untouched! 🎉
inner ids different: True ✅ poorna-freedom


## 5. The Real Test — Outer vs Inner

The single test that distinguishes shallow from deep is to inspect the **inner** objects, not just the outer one.

```text
Shallow: new_box( ptr -> inner1, ptr -> inner2 )   # new box, old links
Deep   : new_box( ptr -> NEW-inner1, ptr -> NEW-inner2 )   # everything fresh
```

If the outer ids differ but any inner id matches, the copy is shallow and the matching parts are shared. If every id differs, the copy is deep. When in doubt, run the `id()` check on the actual structure rather than reasoning about it in your head. This is a common interview question, and the outer-versus-inner check is the answer that satisfies it.

## ⚠️ 5 Common Mistakes (Copy Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Thinking `b = a` is a copy | Aliasing! Copy = `.copy()`/`[:]`/`copy.copy()` |
| 2 | Assuming shallow = "fair-use" on nested | The inner parts are shared! Deep copy `copy.deepcopy` |
| 3 | Relying on `copy.copy()` for nested | It is still shallow — the module or the method, it only copies the outside |
| 4 | Forgetting deepcopy for a tuple-of-lists | Even though a tuple is immutable, the inner lists are shared (live in nb2!) |
| 5 | Passing a list into a function, mutating it, and expecting ORIGINAL-safe | Even when passed, it is still a reference — a direct effect on the caller! (03.04 details) |

> 🎯 **Rule:** FLAT (no inner mutables) → shallow is OK. NESTED (lists inside,
> dicts inside...) → deepcopy. Any doubt? Verify LIVE with `id()`! 🔬

## Summary

- `b = a` is aliasing — one object with two names, sharing the same `id()`.
- Shallow copies (`.copy()`, `[:]`, `copy.copy()`) create a new outer container but share the inner objects.
- On flat lists a shallow copy is enough; on nested structures it leaks mutations to the original.
- `copy.deepcopy()` clones recursively so nothing is shared, at the cost of more time and memory.
- The reliable test is to compare the `id()` of the inner objects, not just the outer container.

Whenever a copy mysteriously changes the original, suspect a shallow copy over nested mutable data.